# PRUEBA TÉCNICA
## Data Scientist — Propiedata
### Sudata · Estimación de precios inmobiliarios

| **Plazo** | 1 semana calendario desde recepción · esfuerzo esperado: ~4 horas de trabajo real |
|-----------|-----------------------------------------------------------------------------------|
| **Modalidad** | Autoorganizada — distribuís el tiempo como prefieras |
| **Entrega** | Fork del repo en GitHub con la rama `prueba-tecnica/<tu-apellido>` |
| **Herramientas** | Python, las librerías que prefieras. MLflow opcional pero suma puntos. |

---

## 1. Contexto

Propiedata es un producto de Sudata para estimación de precios inmobiliarios. Combina scraping de portales públicos, modelado con datos geoespaciales y un servicio de inferencia para usuarios finales. El pipeline actual está implementado en Python (Airflow + scikit-learn + XGBoost + PostGIS) y vive en un repositorio que vas a poder consultar.

Hoy el modelo de venta funciona razonablemente bien, pero el modelo de alquiler tiene desempeño débil (R² muy por debajo de lo aceptable para producción). Además identificamos seis frentes de mejora donde queremos avanzar:

1. Limpieza de datos provenientes de scraping con problemas estructurales
2. Manejo de missing values que hoy obligan a descartar muchas observaciones
3. Identificación de inmuebles únicos cuando un mismo inmueble aparece publicado en varias plataformas o múltiples veces en una misma plataforma
4. Arquitectura de tablas del pipeline (hoy hay desorden entre el output del scraping y el dataset final de entrenamiento)
5. Modelado y experimentación: queremos llevar el R² del modelo de alquileres a 0.8 o más, con RMSE bajo
6. MLOps / CI-CD: hoy no tenemos tracking de experimentos sistemático ni despliegue automatizado

> Esta prueba está diseñada para evaluar tu enfoque sobre estos seis frentes, con foco quirúrgico en dos de ellos.

---

## 2. Objetivo de la prueba

Evaluar tu criterio técnico, capacidad de implementación y capacidad de pensar a nivel arquitectura, sin pedirte resolver todo. La prueba combina dos partes:

- **Parte A — Codeo (~3 horas)**: Implementar mejoras reales sobre dos áreas concretas: limpieza+missings y modelado de alquileres.
- **Parte B — Propuesta escrita (~1 hora)**: Redactar un plan técnico priorizado para las cuatro áreas restantes (matching, arquitectura, modelado avanzado, MLOps).

> No te pedimos resolver todo. Te pedimos mostrar cómo pensás, cómo justificás decisiones, y cómo balanceás rigor con pragmatismo de producto.

---

## 3. Parte A — Codeo (~3 horas)

### 3.1 Datos disponibles

En la carpeta `data/` del fork vas a encontrar tres archivos CSV con listings de alquileres de departamentos, PHs y casas en Buenos Aires. Cada archivo proviene de una plataforma de scraping distinta:

| Archivo | Estructura | Calidad relativa |
|---------|-----------|------------------|
| `listings_plataforma_A.csv` | Tabular completa, ~17 columnas | Formatos sucios, monedas mixtas, valores extremos |
| `listings_plataforma_B.csv` | Tabular reducida, ~17 columnas | Más limpia pero con missings sistemáticos en algunos campos |
| `listings_plataforma_C.csv` | Tabular pobre, ~9 columnas | Información clave en campo de descripción libre |

Las tres plataformas describen el mismo universo de inmuebles, pero un mismo inmueble puede aparecer en varias plataformas y/o varias veces en una misma plataforma. Eso es esperable y forma parte de la realidad del producto.

> No se incluyen reglas pre-impuestas. Los nombres de columnas, formatos, escalas y campos disponibles son distintos en cada plataforma. Parte de tu trabajo es descubrir los problemas y resolverlos.

### 3.2 Tarea 1 — Limpieza, homogeneización y manejo de missings (~1.5 h)

Producir un dataset unificado a partir de las tres plataformas, listo para modelado, justificando las decisiones más relevantes.

#### Entregable
1. Notebook o script en `notebooks/01_limpieza.ipynb` (o `.py` equivalente).
2. CSV o parquet de salida con el dataset unificado y limpio.
3. Resumen breve dentro del notebook con las decisiones más relevantes y por qué (no más de media página).

#### Qué esperamos ver
1. Un schema target unificado y argumentado.
2. Tratamiento explícito de outliers, formatos sucios y campos heterogéneos.
3. Estrategia explícita para missings, con justificación: imputación, descarte, flag de "missing informativo", o combinación.
4. Mínima documentación: decisiones, supuestos, limitaciones.

#### Qué NO esperamos
1. Resolver el matching de inmuebles únicos entre plataformas (eso va en la propuesta escrita).
2. Limpieza perfecta. Esperamos buen criterio aplicado en tiempo limitado.

### 3.3 Tarea 2 — Modelado de alquileres (~1.5 h)

Sobre el dataset limpio de la Tarea 1, entrenar y comparar modelos de estimación de precio mensual de alquiler. Producir métricas reproducibles y trazables.

#### Entregable
1. Notebook o script en `notebooks/02_modelado.ipynb` (o `.py` equivalente).
2. Al menos dos enfoques de modelado comparados (puede ser dos algoritmos, dos sets de features, o variantes razonables).
3. Tabla o salida con las métricas: R² y RMSE en validación, idealmente también MAE y MAPE.
4. Comentario breve sobre qué te llevó al modelo elegido y qué seguirías probando con más tiempo.

#### Bonus que suman
1. Tracking de experimentos en MLflow (no obligatorio pero altamente valorado).
2. Validación robusta (cross-validation, no solo train-test split).
3. Análisis de errores: dónde falla el modelo y por qué.
4. Uso de variables geoespaciales derivadas de las coordenadas.

#### Sobre métricas objetivo
En producción apuntamos a **R² ≥ 0.8** con RMSE bajo en alquileres. En esta prueba no exigimos llegar a 0.8 — el objetivo es que muestres tu proceso de experimentación y tu criterio. Un R² de 0.7 con buen análisis vale más que un 0.85 sin justificación.

---

## 4. Parte B — Propuesta escrita (~1 hora)

Después de explorar el dataset y el repositorio, redactá un documento `PROPUESTA.md` en la raíz del fork con tu plan técnico para las cuatro áreas restantes:

| Área | Qué queremos saber |
|------|-------------------|
| **4.1 Matching de inmuebles únicos** | ¿Cómo identificarías que dos publicaciones (intra o entre plataformas) corresponden al mismo inmueble físico? Estrategia, features, métricas de evaluación, riesgos. |
| **4.2 Arquitectura de tablas** | ¿Qué tablas/capas tendría el pipeline ideal desde el output del scraping hasta el dataset de entrenamiento y la capa de UI? Diagrama o descripción de capas, criterios de versionado. |
| **4.3 Modelado avanzado para R²=0.8** | ¿Qué probarías para alcanzar el objetivo? Familias de modelos, features espaciales adicionales, segmentación, regularización, ensembles. Priorizá por impacto esperado vs costo. |
| **4.4 MLOps / CI-CD** | ¿Cómo organizarías tracking de experimentos, registro y promoción de modelos, monitoreo de drift y despliegue automatizado? Stack mínimo viable y stack ideal. |

#### Formato esperado
1. Una sección por área, ~250-400 palabras cada una.
2. Cada sección con: diagnóstico breve, propuesta concreta, esfuerzo estimado y riesgos.
3. Priorización general al final: si tuvieses 1 mes, ¿en qué orden atacarías estos 4 frentes y por qué?

> Valoramos pragmatismo. Una propuesta realista de 6 semanas es mejor que una propuesta perfecta de 6 meses.

---

## 5. Entrega

### 5.1 Estructura esperada del fork

├── data/ ← datasets recibidos (no commitear si son grandes)
├── notebooks/
│ ├── 01_limpieza.ipynb ← Tarea 1
│ └── 02_modelado.ipynb ← Tarea 2
├── output/
│ └── dataset_unificado.parquet ← salida de Tarea 1
├── PROPUESTA.md ← Parte B
├── README_ENTREGA.md ← cómo correr tu código en 1 minuto
└── requirements_entrega.txt ← deps adicionales si las hay
### 5.2 Reproducibilidad
Tu código debe poder correrse de punta a punta. Un `README_ENTREGA.md` con los pasos para reproducir tus resultados es parte de la evaluación.

### 5.3 Cómo entregar
1. Crear un fork del repositorio que te compartimos.
2. Trabajar en una rama `prueba-tecnica/<tu-apellido>`.
3. Cuando termines, mandar el link del fork (rama incluida) por email respondiendo al hilo de la convocatoria.

---

## 6. Cómo te vamos a evaluar

Estos son los pesos que vamos a usar al revisar tu entrega:

| Criterio | Peso | Qué miramos |
|----------|------|-------------|
| Calidad del código y reproducibilidad | 20% | Estructura clara, código que corre, naming consistente, README que funciona |
| Justificación de decisiones (limpieza/missings) | 20% | Diagnóstico explícito, criterio detrás de cada decisión, reconocimiento de trade-offs |
| Performance del modelo + rigor experimental | 25% | Métricas honestas, validación robusta, comparación de enfoques, análisis de errores |
| Calidad de la propuesta escrita (4 áreas) | 25% | Pragmatismo, especificidad, priorización razonada, lectura del problema real |
| MLOps básico (MLflow, estructura, README) | 10% | Tracking de experimentos, repo navegable, documentación mínima |

### Lo que vamos a valorar especialmente
1. Pensar como **product DS**, no como Kaggle DS: criterio sobre qué vale la pena resolver y qué no.
2. Honestidad sobre lo que no llegaste a hacer y por qué.
3. Decisiones explicables — más importante el **por qué** que el **qué**.
4. Pragmatismo: 4 horas alcanzan para mostrar criterio, no para resolver todo. Que se note que sabés priorizar.

### Lo que NO valoramos
1. Notebooks de 80 celdas con 200 gráficos exploratorios.
2. Modelos perfectos sin justificación.
3. Pasarse del tiempo esperado para sacar décimas de R².

---

> Cualquier duda durante la prueba, podés mandar un email al hilo de la convocatoria.
> 
> **¡Suerte! Estamos para responder lo que necesites.** 🚀